# Information Crystallization — Seed Phase on GPT-2

**What this notebook does:** runs Phase 1 (seed) of the Information Crystallization algorithm on GPT-2 small (124M params) for a stratified sample of input sequences. For each sequence, it finds:

1. $j^* = \arg\max_j |g_j|$ where $g = \nabla_\theta \log P_\theta(\hat{x} \mid X)$ — the single most gradient-sensitive parameter
2. $t^* = \arg\max_t \|\partial f_\theta / \partial x_t\|$ — the single most gradient-sensitive input position

**What this notebook does NOT do:** Phases 2–4 (growth, prune, verify) are not implemented here. This is a sanity check on the seed step only — the cheapest step of the algorithm.

**Authorship note:** I wrote this notebook with AI assistance for scaffolding and debugging. I ran every cell myself, verified the numbers, and can defend every design choice. — Saqib Nazir Bhat

## Setup

If running on Colab, uncomment the install cell. On a local machine with PyTorch already installed, it's a no-op.

In [ ]:
# !pip install -q transformers torch matplotlib pandas

In [ ]:
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'gpt2'  # 124M params — smallest and fastest
print(f'Device: {DEVICE}')

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()  # no dropout; deterministic forward

# Sanity check: total parameter count
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_NAME}, total parameters: {n_params:,}')

## Sample sequences (stratified by query type)

The paper proposes stratification by four query types. Here I include a small hand-curated sample — big enough to show a distribution, small enough to run on CPU in a few minutes.

In [ ]:
SEQUENCES = [
    # Syntactic continuation
    ('syntactic', 'The quick brown fox jumps over the lazy'),
    ('syntactic', 'She opened the door and walked into the'),
    ('syntactic', 'After the rain stopped, the children ran outside to'),
    ('syntactic', 'He picked up the book and started to'),
    ('syntactic', 'The best way to learn programming is to'),

    # Factual recall
    ('factual', 'The capital of France is'),
    ('factual', 'The author of Harry Potter is'),
    ('factual', 'The largest planet in our solar system is'),
    ('factual', 'The chemical symbol for gold is'),
    ('factual', 'Albert Einstein developed the theory of'),

    # Negation-heavy (tests whether position-1 negation dominates)
    ('negation', 'Despite all the warnings, the captain decided to'),
    ('negation', 'Not everyone agreed, but the team chose to'),
    ('negation', 'Although it was raining, they went for a'),
    ('negation', 'Without further delay, the meeting will'),

    # Long-range syntactic (subject at start determines later form)
    ('long-range', 'The scientists who discovered the new particle yesterday have'),
    ('long-range', 'The man standing next to the blue car with the broken window is'),
    ('long-range', 'The cats in the garden behind the old wooden fence are'),
]

print(f'Total sequences: {len(SEQUENCES)}')
print(pd.Series([t for t, _ in SEQUENCES]).value_counts())

## The seed-phase computation

For a single sequence $X$:

1. Run the forward pass. Let $\ell = f_\theta(X)$ be the final-position logits.
2. $\hat{x} = \arg\max_v \ell_v$ — the model's prediction.
3. Compute $\log P_\theta(\hat{x} \mid X) = \text{log\_softmax}(\ell)_{\hat{x}}$.
4. Call `.backward()`. For each parameter tensor $\theta_i$, this gives $\partial \log P_\theta(\hat{x} \mid X) / \partial \theta_i$.
5. Find the single scalar with the largest absolute gradient. That's $j^*$.

For the input seed $t^*$: I take the gradient w.r.t. the token embeddings of the input (`inputs_embeds`) and take the L2 norm per position.

**Design choices I am making here, and why:**

- **Log-probability, not raw logit.** The paper talks about preserving argmax, which only requires $\delta > 0$. But for ranking parameters at the seed step, log-probability is the standard interpretability target (used in saliency, integrated gradients, etc.) and gives the same ordering as logits in practice for the winning class. Raw-logit gradient is a reasonable alternative; ranks would be similar.
- **L2 norm on embedding gradients.** The paper writes $\|\partial f_\theta / \partial x_t\|$ without specifying a norm. L2 is the standard; L1 would also be defensible. I report both at the end so it's transparent.
- **GPT-2 small, not GPT-2 medium.** Cheapest model that still has the residual-stream structure the paper assumes. Findings here are suggestive, not conclusive for larger models.
- **Single sequence at a time, not batched.** Backward pass w.r.t. input embeddings needs per-sequence gradients; batching adds complexity without speeding up on a dataset this small.

In [ ]:
def seed_phase(text: str):
    """Run the seed step on a single input sequence.
    
    Returns a dict with:
      - x_hat: predicted token string
      - x_hat_token_id: predicted token id
      - logit_margin: the delta = winning logit - 2nd winning logit
      - param_seed_name: name of the parameter tensor holding j*
      - param_seed_layer: which transformer block (0..n_layers-1 or 'unembedding' / 'embedding')
      - param_seed_grad: |g_{j*}|
      - input_seed_position: t* (0-indexed position in input)
      - input_seed_token: the token at position t*
      - input_seed_grad_l2: L2 norm of the seed position's embedding gradient
      - input_seed_grad_l1: L1 norm (for transparency)
      - seq_length: number of tokens
    """
    # --- tokenize ---
    input_ids = tokenizer.encode(text, return_tensors='pt').to(DEVICE)
    seq_len = input_ids.shape[1]
    tokens_str = [tokenizer.decode([t]) for t in input_ids[0].tolist()]
    
    # --- forward pass with embeddings that require grad ---
    # We go through inputs_embeds so we can backprop to the per-token embedding
    embeds = model.transformer.wte(input_ids).detach().clone()
    embeds.requires_grad_(True)
    
    # Clear parameter grads
    for p in model.parameters():
        if p.grad is not None:
            p.grad.zero_()
    model.zero_grad()
    
    outputs = model(inputs_embeds=embeds)
    logits = outputs.logits[0, -1, :]  # [vocab_size]
    
    # --- x_hat = argmax ---
    x_hat_id = int(logits.argmax().item())
    x_hat_str = tokenizer.decode([x_hat_id])
    
    # --- logit margin ---
    top2 = torch.topk(logits, 2)
    delta = (top2.values[0] - top2.values[1]).item()
    
    # --- log P(x_hat | X) ---
    log_probs = F.log_softmax(logits, dim=-1)
    target = log_probs[x_hat_id]
    
    # --- backward ---
    target.backward()
    
    # --- find parameter j* ---
    best_val, best_name = 0.0, None
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        absmax = p.grad.abs().max().item()
        if absmax > best_val:
            best_val = absmax
            best_name = name
    
    # --- classify seed location ---
    seed_layer = classify_param_layer(best_name)
    
    # --- find input position t* ---
    # embeds.grad has shape [1, seq_len, embed_dim]
    per_pos_l2 = embeds.grad[0].norm(p=2, dim=-1)  # [seq_len]
    per_pos_l1 = embeds.grad[0].norm(p=1, dim=-1)
    t_star = int(per_pos_l2.argmax().item())
    
    return {
        'text': text,
        'seq_length': seq_len,
        'x_hat': x_hat_str,
        'x_hat_token_id': x_hat_id,
        'logit_margin': delta,
        'param_seed_name': best_name,
        'param_seed_layer': seed_layer,
        'param_seed_grad': best_val,
        'input_seed_position': t_star,
        'input_seed_position_from_end': seq_len - 1 - t_star,
        'input_seed_token': tokens_str[t_star],
        'input_seed_grad_l2': per_pos_l2[t_star].item(),
        'input_seed_grad_l1': per_pos_l1[t_star].item(),
    }

def classify_param_layer(name):
    """Map a GPT-2 parameter name to a location class."""
    if name is None:
        return 'unknown'
    if 'wte' in name:
        return 'embedding'
    if 'wpe' in name:
        return 'position_embedding'
    if 'lm_head' in name:
        return 'unembedding'  # GPT-2 ties this to wte by default
    if '.h.' in name:
        # e.g. transformer.h.5.mlp.c_fc.weight
        block_idx = int(name.split('.h.')[1].split('.')[0])
        return f'block_{block_idx}'
    if 'ln_f' in name:
        return 'final_layernorm'
    return 'other'

# Sanity check: run one sequence
result = seed_phase('The capital of France is')
for k, v in result.items():
    print(f'{k:30s}: {v}')

## Run across all sequences

In [ ]:
records = []
for query_type, text in SEQUENCES:
    r = seed_phase(text)
    r['query_type'] = query_type
    records.append(r)
    print(f'[{query_type:10s}] "{text}" -> "{r["x_hat"]}" | seed={r["param_seed_layer"]} | t*={r["input_seed_position"]}/{r["seq_length"]-1} ({r["input_seed_token"]!r})')

df = pd.DataFrame(records)
df

## Finding 1: Where does $j^*$ live?

The paper's $S_L$ conjecture predicts that the parameter most sensitive to the winning-logit probability lives in the **final transformer block** or the **unembedding projection**. Let's check.

In [ ]:
layer_counts = df['param_seed_layer'].value_counts()
print('Seed-parameter location distribution:')
print(layer_counts)

# Group by 'final block / unembedding' vs 'middle / early'
N_LAYERS = model.config.n_layer  # GPT-2 small = 12
def is_final_region(loc):
    return loc in ('unembedding', 'embedding', 'final_layernorm') or loc == f'block_{N_LAYERS-1}'

df['seed_in_final_region'] = df['param_seed_layer'].apply(is_final_region)
print(f"\nFraction of sequences with seed in final region (unembedding / final block): "
      f"{df['seed_in_final_region'].mean():.1%}")

In [ ]:
# Stratified view: does factual-recall seed live in middle layers (ROME hypothesis)?
by_type = df.groupby('query_type')['param_seed_layer'].value_counts()
print('Seed location by query type:')
print(by_type)

## Finding 2: Where does $t^*$ live?

The paper conjectures $T = \{x_{n-k}, \ldots, x_n\} \cup T_{\text{sal}}$ — suffix plus attention-salient earlier positions. The suffix conjecture predicts $t^*$ should typically be near the end of the sequence. The counter-example is long-range / negation-heavy sequences where the key information lives at position 1.

In [ ]:
# Where in the sequence does t* typically land?
df['seed_is_last'] = df['input_seed_position'] == (df['seq_length'] - 1)
df['seed_is_first_3'] = df['input_seed_position'] < 3

print('Input-seed position statistics:')
print(f"  seed = last token:         {df['seed_is_last'].mean():.1%}")
print(f"  seed in first 3 tokens:    {df['seed_is_first_3'].mean():.1%}")
print(f"  mean distance from end:    {df['input_seed_position_from_end'].mean():.2f} tokens")
print()
print('By query type (mean distance from end):')
print(df.groupby('query_type')['input_seed_position_from_end'].mean())

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: param-seed layer distribution
layer_counts.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title(f'Seed parameter j* — location\n(GPT-2 small, {N_LAYERS} transformer blocks)')
axes[0].set_xlabel('Count')

# Right: input-seed position (distance from end)
axes[1].hist(df['input_seed_position_from_end'], bins=range(0, df['seq_length'].max()+1),
             color='steelblue', edgecolor='black')
axes[1].set_title('Input seed t* — distance from end')
axes[1].set_xlabel('n - 1 - t* (0 = last token)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('seed_phase_results.png', dpi=120)
plt.show()

## What this does and does not show

**Shows:**
- Whether the seed-parameter $j^*$ localizes in the final transformer region, consistent with $S_L$ conjecture, on this sample.
- Whether the seed-input-position $t^*$ is suffix-concentrated or distributed, stratified by query type.
- A sanity check that the algorithm's first step produces interpretable, reproducible outputs on a real model.

**Does not show:**
- Whether the full (S, T) set is actually small (Phases 2–4 unimplemented).
- Whether the Sufficient Condition (Lipschitz-like perturbation bound) holds.
- Anything about larger models — GPT-2 small has only 12 layers; distributed representations may look different at scale.
- Statistical significance — $N = 17$ sequences is suggestive, not conclusive. A proper study would use the full 1,000-sequence protocol stratified more carefully.

## Reproducibility

- Model: `gpt2` via HuggingFace `transformers`, loaded in eval mode
- Runtime: CPU ~2–4 min for all 17 sequences; GPU well under 30 sec
- Deterministic: no sampling, pure forward + backward
- All code is in this notebook; no external datasets required